[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/hyperparameters/notebooks/ex2_tuning.ipynb)

# Tuning hyperparameters with a library

In the lecture a hyperparameter was anything you have to choose before
learning starts. Today you choose them with a library instead of by hand, on
three black boxes of growing price:

| part | the black box | one evaluation costs |
|---|---|---|
| 1 | a treatment regimen, dose and interval | a measurement (here: microseconds) |
| 2 | an SVM on breast-cancer biopsies | a 5-fold cross-validation, 0.04 s |
| 3 | a small convolutional network on leaf photographs | a whole training, 3-18 s |

The method never changes: **write a function that returns one number, say in
what box its parameters live, hand both to the library.** What changes is the
price of one call - and that is what decides how much you can afford to search.

There are **five holes** marked `# TODO`; everything else is prepared. Colab is
enough for all of it, and part 3 is noticeably faster with a GPU:
*Runtime - Change runtime type - T4 GPU*.

In [ ]:
%pip install -q bayesian-optimization optuna

# Part 1 - a black box and a library

An expensive experiment: you set a dose and an interval between cycles, and a
week later one number comes back - and it is one patient's response, not the
truth. No formula, no derivative, and twenty measurements is all you get.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- An expensive black box ------------------------------------------------
# A treatment regimen goes in - a dose d [mg/m2] per cycle and an interval T
# [days] between cycles - and one noisy score comes out. The same simulator as
# in the Bayesian optimization lab. Do not read the formula: in practice there
# is a week of measurement behind this line, and no formula at all.

K, D0, W, ALPHA, RHO, T_REF = 0.55, 85.0, 6.0, 40.0, 0.05, 14.0
BETA, GAMMA = 1.6312416378843562, 20.713627157106966
BOUNDS = {"dose": (20.0, 100.0), "interval": (10.0, 35.0)}
SIGMA_NOISE = 2.0                         # a patient does not respond twice the same
rng = np.random.default_rng(0)


def score(d, T):
    """The true score. AVAILABLE ONLY FOR CHECKING AT THE END, not for searching."""
    d, T = np.asarray(d, float), np.asarray(T, float)
    I = d / T
    return (100.0 * (1.0 - np.exp(-K * I))
            - ALPHA / (1.0 + np.exp(-(d - D0) / W))
            - BETA * I**2
            - GAMMA * (21.0 / T)
            - RHO * np.maximum(0.0, T - T_REF) ** 2)


def measure(d, T):
    """ONE EXPENSIVE EVALUATION: the score of one patient, noise included."""
    return float(score(d, T) + rng.normal(0.0, SIGMA_NOISE))


print(f"one measurement at (60, 21): {measure(60.0, 21.0):.2f} points")
print(f"and again, the same regimen: {measure(60.0, 21.0):.2f} points")

The same regimen gives a different number twice; that is the noise
you are searching through.

Now the library. `bayes_opt` (the package `bayesian-optimization`) needs
exactly two things - a function that returns a number to **maximize**, and the
box its arguments live in.

In [ ]:
from bayes_opt import BayesianOptimization

# TODO 1: hand the black box to the library. Two things are yours to write:
#
#   1) `objective(dose, interval)` - the function `bayes_opt` will maximize.
#      It takes the parameters BY NAME (that is the whole interface) and
#      returns one number, the larger the better;
#   2) `pbounds` - the box to search in, as {"name": (low, high)}; `BOUNDS`
#      above is already in that shape.
#
# Then run 5 random points and 15 steps of the acquisition:
#     optimizer.maximize(init_points=5, n_iter=15)

def objective(dose, interval):
    ...


optimizer = BayesianOptimization(f=..., pbounds=..., random_state=0, verbose=0)
optimizer.maximize(init_points=5, n_iter=15)

best = optimizer.max
print(f"best measured: {best['target']:.2f} points at "
      f"dose {best['params']['dose']:.1f} mg/m2, "
      f"interval {best['params']['interval']:.1f} days")

We happen to know the true landscape here, which in real life we
never do. It is worth looking at where the twenty measurements went.

In [ ]:
# What it did with the twenty measurements - on the landscape it never saw.
visited = np.array([[r["params"]["dose"], r["params"]["interval"]]
                    for r in optimizer.res])
dd = np.linspace(*BOUNDS["dose"], 200)
tt = np.linspace(*BOUNDS["interval"], 200)
landscape = score(dd[None, :], tt[:, None])

plt.figure(figsize=(6, 4))
plt.contourf(dd, tt, landscape, levels=25, cmap="viridis")
plt.colorbar(label="true score [points]")
plt.plot(visited[:5, 0], visited[:5, 1], "wo", label="5 random")
plt.plot(visited[5:, 0], visited[5:, 1], "r.", markersize=12, label="15 by the model")
plt.plot(60, 21, "w*", markersize=16, label="the true optimum")
plt.xlabel("dose [mg/m2]"); plt.ylabel("interval [days]"); plt.legend()
plt.show()

true_here = float(score(best["params"]["dose"], best["params"]["interval"]))
print(f"the true optimum is 42.13 points at (60 mg/m2, 21 days)")
print(f"the recommended regimen is truly worth {true_here:.2f} points, "
      f"but what the search saw was one noisy measurement: {best['target']:.2f}")

# Part 2 - the same three lines on a model

A hyperparameter search is the same black box, only the evaluation is a
training. The model: a support-vector machine with an RBF kernel on the
breast-cancer biopsies, with the two hyperparameters everybody tunes - `C` and
`gamma`.

**One evaluation is a whole cross-validation**, not a single fit: five
trainings on four fifths of the data each. The test part is set aside and
touched once, at the end.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

data = load_breast_cancer()
x_train, x_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=0)


def cross_validated_accuracy(C, gamma):
    """ONE EVALUATION of a configuration: a whole 5-fold cross-validation.

    The standardization is INSIDE the pipeline on purpose - in a
    cross-validation it may only ever see the training folds.
    """
    model = make_pipeline(StandardScaler(), SVC(C=C, gamma=gamma))
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    return 100 * cross_val_score(model, x_train, y_train, cv=folds).mean()


print(f"{len(y_train)} training and {len(y_test)} test biopsies, "
      f"{x_train.shape[1]} features")
print(f"default C = 1, gamma = 1/30: {cross_validated_accuracy(1.0, 1 / 30):.2f} %")

In [ ]:
# TODO 2: the same three lines as in part 1, pointed at the model.
#
# One difference, and it is the point of the block: C and gamma are searched
# on a LOGARITHMIC scale. Their sensible range spans seven orders of magnitude,
# so the search works with log10 C and log10 gamma and the objective raises 10
# to them. Budget: 5 + 15 evaluations again.

space = {"log_C": (-3.0, 4.0), "log_gamma": (-4.0, 1.0)}


def svm_objective(log_C, log_gamma):
    ...


search = ...
search.maximize(init_points=5, n_iter=15)

tuned_C = 10**search.max["params"]["log_C"]
tuned_gamma = 10**search.max["params"]["log_gamma"]
print(f"tuned: {search.max['target']:.2f} % at C = {tuned_C:.3g}, "
      f"gamma = {tuned_gamma:.3g}")

In [ ]:
# The test part, touched once: the tuned model against the default one.
tuned = make_pipeline(StandardScaler(), SVC(C=tuned_C, gamma=tuned_gamma)).fit(x_train, y_train)
default = make_pipeline(StandardScaler(), SVC(C=1.0, gamma=1 / 30)).fit(x_train, y_train)

print(f"tuned   {100 * tuned.score(x_test, y_test):.2f} % on the test part")
print(f"default {100 * default.score(x_test, y_test):.2f} % on the test part")

Write down both numbers and compare them with the default. The
tuning gained about half a point of cross-validated accuracy - and the test
part does not confirm it. That is not a bug in the search: a difference of half
a point on 426 samples is smaller than the noise, and the search reports the
**best of twenty** noisy numbers, which is optimistic by construction. The
honest conclusion here is *the default was already good enough*.

# Part 3 - a network, a train function, and two libraries

Four species of leaf from a photograph: 700 training images of 96 x 96 pixels,
a small convolutional network, 8 epochs. This is where tuning starts to pay -
and where one evaluation starts to hurt.

[Download the dataset](https://github.com/tomasvicar/AUI-public/blob/master/labs/hyperparameters/data/ex08_leaves_images.zip) ·
[Dataset provenance](https://github.com/tomasvicar/AUI-public/blob/master/labs/hyperparameters/data/README.md)
The archive is included in this course repository; the notebook downloads it
automatically. For an offline run, place it beside the notebook as `leaves.zip`.

In [ ]:
import glob, os, urllib.request, zipfile

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset

URL = "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/hyperparameters/data/ex08_leaves_images.zip"
CLASSES = ["apple", "cherry", "chestnut", "maple"]
SIZE = 48                      # the photographs are 96 x 96; smaller = faster

if not os.path.exists("leaves"):
    if not os.path.isfile("leaves.zip"):
        urllib.request.urlretrieve(URL, "leaves.zip")
    zipfile.ZipFile("leaves.zip").extractall(".")


def load_split(split):
    """One split as a tensor (n, 3, SIZE, SIZE) - 820 small images fit in memory."""
    files = sorted(glob.glob(f"leaves/{split}/*/*.jpg"))
    images = np.stack([np.asarray(Image.open(f).convert("RGB").resize((SIZE, SIZE)),
                                  dtype=np.float32) / 255.0 for f in files])
    labels = [CLASSES.index(f.split("/")[2]) for f in files]
    return torch.from_numpy(images).permute(0, 3, 1, 2), torch.tensor(labels)


x_train_img, y_train_img = load_split("train")
x_rest, y_rest = load_split("val")

# The 120 held-out photographs are split in half: the VALIDATION half is what
# every trial of the search sees, the TEST half is looked at once, at the end.
order = torch.randperm(len(y_rest), generator=torch.Generator().manual_seed(0))
x_valid_img, y_valid_img = x_rest[order[:60]], y_rest[order[:60]]
x_test_img, y_test_img = x_rest[order[60:]], y_rest[order[60:]]

print(f"train {len(y_train_img)}, validation {len(y_valid_img)}, test {len(y_test_img)}")
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for ax, i in zip(axes, torch.randperm(len(y_train_img))[:8]):
    ax.imshow(x_train_img[i].permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[y_train_img[i]], fontsize=9)
    ax.axis("off")
plt.show()

The network: three convolutional blocks and one linear layer,
about 24 000 parameters.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on", device)


class LeafCNN(nn.Module):
    """Three blocks of conv 3x3 + ReLU + max pool, then one linear layer."""

    def __init__(self, n_filters=16, dropout=0.0):
        super().__init__()
        layers, channels = [], 3
        for block in range(3):
            width = n_filters * 2**block
            layers += [nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)]
            channels = width
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                        nn.Dropout(dropout), nn.Linear(channels, 4))

    def forward(self, x):
        return self.classifier(self.features(x))


def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        return float((model(x.to(device)).argmax(1).cpu() == y).float().mean())

And one training, with the hyperparameters written in by hand -
the way everybody starts, and the way nobody should finish.

In [ ]:
# --- one training, with the hyperparameters written in by hand --------------
learning_rate, weight_decay, n_filters, dropout, batch_size = 1e-3, 0.0, 16, 0.0, 32
n_epochs = 8                       # fixed part of the budget, not tuned

torch.manual_seed(0)
model = LeafCNN(n_filters, dropout).to(device)
optimizer_nn = torch.optim.Adam(model.parameters(), lr=learning_rate,
                                weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()
loader = DataLoader(TensorDataset(x_train_img, y_train_img),
                    batch_size=batch_size, shuffle=True,
                    generator=torch.Generator().manual_seed(0))

for epoch in range(n_epochs):
    model.train()
    for images, labels in loader:          # the five lines of learning
        images, labels = images.to(device), labels.to(device)
        optimizer_nn.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer_nn.step()
    print(f"epoch {epoch + 1}  validation accuracy "
          f"{accuracy(model, x_valid_img, y_valid_img):.3f}")

default_accuracy = accuracy(model, x_valid_img, y_valid_img)

### The training as a function

Nothing can be searched until the training is a function: **hyperparameters in,
one number out**. This is the whole refactoring step, and it is the part that
is yours to do.

In [ ]:
# TODO 3: wrap the cell above into a function.
#
# Everything that was written in by hand becomes an argument; everything that
# must not change - the data, the epochs, the seed - stays fixed inside. What
# comes back is ONE number, the validation accuracy. That is the black box of
# part 1, only expensive for real.
#
# Move the cell above into the body, replace the five constants by the
# arguments and return the accuracy instead of printing it. The data to score
# on is an argument too, so that the test half can be passed in at the end.

def train_and_validate(learning_rate=1e-3, weight_decay=0.0, n_filters=16,
                       dropout=0.0, batch_size=32, n_epochs=8, seed=0,
                       x_valid=x_valid_img, y_valid=y_valid_img):
    ...


print(f"the default configuration again: {train_and_validate():.3f}")

### Searching it with `bayes_opt`

The same library as in parts 1 and 2 - but a network has hyperparameters that
a box of real numbers does not describe well, and you meet both kinds here.

In [ ]:
# TODO 4: search four hyperparameters with bayes_opt. Budget 4 + 8 trainings,
# which is a minute or two.
#
# Two things the box of real numbers cannot express, and you have to:
#   - the learning rate and the weight decay live on a LOG scale, so search
#     log10 of them and raise 10 to it inside the objective;
#   - `n_filters` is an INTEGER; bayes_opt only knows real numbers, so round it
#     (and note that the model it builds does not know you did).

nn_space = {"log_learning_rate": (-4.0, -1.5), "log_weight_decay": (-6.0, -2.0),
            "n_filters": (8.0, 24.0), "dropout": (0.0, 0.5)}


def nn_objective(log_learning_rate, log_weight_decay, n_filters, dropout):
    ...


nn_search = ...
nn_search.maximize(init_points=4, n_iter=8)

found = nn_search.max["params"]
print(f"bayes_opt: {nn_search.max['target']:.3f} "
      f"(the default was {default_accuracy:.3f})")
print(f"  learning rate {10**found['log_learning_rate']:.1e}, "
      f"weight decay {10**found['log_weight_decay']:.1e}, "
      f"{int(round(found['n_filters']))} filters, dropout {found['dropout']:.2f}")

### The same search in Optuna

Optuna asks for one parameter at a time and is told **what kind** it is: a
float on a log scale, an integer, or a choice from a list. The choice is what
`bayes_opt` could not take at all.

In [ ]:
import optuna

# TODO 5: the same search in Optuna, plus one parameter bayes_opt could not
# take - the batch size, which is a CHOICE of 16, 32 or 64.
#
# Optuna asks the trial for every parameter, and says what kind it is:
#     trial.suggest_float("learning_rate", 1e-4, 3e-2, log=True)
#     trial.suggest_int("n_filters", 8, 24)
#     trial.suggest_categorical("batch_size", [16, 32, 64])
# No logarithms and no rounding by hand - the sampler knows the shape of each
# parameter, and that is the whole difference. Same budget: 12 trials.

def optuna_objective(trial):
    ...


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(optuna_objective, n_trials=12)

print(f"optuna: {study.best_value:.3f}")
print(study.best_params)

In [ ]:
# What the twelve trials looked like: the best value so far, trial by trial.
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.show()

### The number you would report

The searched accuracy is the best of twelve tries on 60 photographs. The test
half has taken no part in any of it.

In [ ]:
# The 60 test photographs, once: the winning configuration retrained and
# scored on data that took no part in the search.
on_test = train_and_validate(**study.best_params, x_valid=x_test_img, y_valid=y_test_img)

print(f"default configuration:  validation {default_accuracy:.3f}")
print(f"tuned configuration:    validation {study.best_value:.3f}  "
      f"test {on_test:.3f}")

Tuning was worth five to ten points of accuracy here - a real gain,
unlike part 2. The two numbers at the bottom will not agree, in one direction
or the other, and the reason is worth more than either of them: an accuracy
measured on 60 photographs has a standard error of about **6 points**. The
search chose the best of twelve such numbers, so the validation one is
optimistic by construction; the test one is unbiased but just as noisy. Both
belong in a report - the first says what the search found, the second says what
it is worth - and neither justifies a sentence about a difference of three
points.

## What to take away

1. **The library is three lines.** A function returning one number, a box, a
   budget - the same three lines for a simulator, an SVM and a network.
2. **The price of one evaluation decides everything else.** A millisecond
   allows thousands of trials; a training allows a dozen, and then the search
   space matters more than the sampler.
3. **The scale of a parameter is part of the search space.** Learning rates and
   `C` are searched in logarithms, or most of the budget is spent in a range
   nobody would use.
4. **`bayes_opt` knows only boxes of real numbers.** Integers have to be
   rounded and choices cannot be expressed at all; Optuna asks for each
   parameter by its kind, which is why it wins as soon as the space is mixed.
5. **The tuned number is the best of many, so it is optimistic.** It is not the
   number you report - part 2 is the cleanest example of that in this lab.

---

*The numbers this notebook is compared against are computed by
[`code/classic_model.py`](https://github.com/tomasvicar/AUI-public/blob/master/labs/hyperparameters/code/classic_model.py) and
[`code/leaf_cnn.py`](https://github.com/tomasvicar/AUI-public/blob/master/labs/hyperparameters/code/leaf_cnn.py). The notebook itself is built by
[`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/hyperparameters/code/build_notebooks.py) - editing the
`.ipynb` by hand gets overwritten.*